# Problem 1 - RNN, LSTM and GRU

All code for this problem is in this notebook. The student-owned cell implementations remain TODOs as required by the assignment.

In [ ]:
from pathlib import Path
import random, time, urllib.request, zipfile, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import ucimlrepo

ROOT = Path.cwd().parent if Path.cwd().name == 'problem1' else Path.cwd()
FIGURES = ROOT / 'problem1' / 'figures'; FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / 'results.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHANNELS = ('body_acc_x','body_acc_y','body_acc_z','body_gyro_x','body_gyro_y','body_gyro_z','total_acc_x','total_acc_y','total_acc_z')
HAR_URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
print(DEVICE)

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def make_adding(n, T, seed):
    rng=np.random.default_rng(seed); x=np.zeros((n,T,2),dtype=np.float32); x[:,:,0]=rng.random((n,T))
    i1=rng.integers(0,T//2,n); i2=rng.integers(T//2,T,n)
    x[np.arange(n),i1,1]=1; x[np.arange(n),i2,1]=1
    return x,(x[np.arange(n),i1,0]+x[np.arange(n),i2,0]).astype(np.float32)

def assert_adding(x,y):
    m=x[:,:,1]; assert np.all(m.sum(1)==2); assert np.all(m[:,:x.shape[1]//2].sum(1)==1); assert np.all(m[:,x.shape[1]//2:].sum(1)==1)
    np.testing.assert_allclose((x[:,:,0]*m).sum(1),y,atol=1e-6,rtol=0)

for T in (50,100,200,400): assert_adding(*make_adding(128,T,0))
print('adding-problem checks passed')

In [ ]:
def download_har(data_root=ROOT/'data'/'har'):
    data_root.mkdir(parents=True,exist_ok=True); archive=data_root/'har.zip'
    if not archive.exists(): urllib.request.urlretrieve(HAR_URL,archive)
    with zipfile.ZipFile(archive) as z: z.extractall(data_root)
    found=list(data_root.rglob('UCI HAR Dataset'))
    if not found: raise FileNotFoundError('UCI HAR Dataset not found')
    return found[0]

def load_har(validation_seed=0):
    base=download_har()
    def read(split):
        d=base/split/'Inertial Signals'; x=np.stack([np.loadtxt(d/f'{c}_{split}.txt') for c in CHANNELS],axis=-1).astype('float32')
        y=np.loadtxt(base/split/f'y_{split}.txt',dtype='int64')-1; s=np.loadtxt(base/split/f'subject_{split}.txt',dtype='int64'); return x,y,s
    x,y,s=read('train'); xt,yt,st=read('test'); held=np.random.default_rng(validation_seed).choice(np.unique(s),4,replace=False); vm=np.isin(s,held); tm=~vm
    mean=x[tm].mean((0,1),keepdims=True); std=np.maximum(x[tm].std((0,1),keepdims=True),1e-6); norm=lambda a:((a-mean)/std).astype('float32')
    return {'train':(norm(x[tm]),y[tm],s[tm]),'validation':(norm(x[vm]),y[vm],s[vm]),'test':(norm(xt),yt,st),'mean':mean,'std':std,'held_out':held}

# Uncomment to download/load: har=load_har(validation_seed=0)

In [ ]:
class MyRNNCell(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__(); self.input_dim=input_dim; self.hidden_dim=hidden_dim
        self.W_x=nn.Parameter(torch.empty(input_dim,hidden_dim)); self.W_h=nn.Parameter(torch.empty(hidden_dim,hidden_dim)); self.b=nn.Parameter(torch.zeros(hidden_dim))
        nn.init.xavier_uniform_(self.W_x); nn.init.orthogonal_(self.W_h)
    def forward(self,x_t,h_prev):
        return torch.tanh(x_t @ self.W_x + h_prev @ self.W_h + self.b)

class MyLSTMCell(nn.Module):
    def __init__(self,input_dim,hidden_dim,forget_bias=1.0):
        super().__init__(); self.input_dim=input_dim; self.hidden_dim=hidden_dim; self.forget_bias=forget_bias
        self.W=nn.Parameter(torch.empty(input_dim+hidden_dim,4*hidden_dim)); self.b=nn.Parameter(torch.zeros(4*hidden_dim)); nn.init.xavier_uniform_(self.W)
        with torch.no_grad(): self.b[hidden_dim:2*hidden_dim].fill_(forget_bias)
    def forward(self,x_t,state):
        h_prev,c_prev=state; gates=torch.cat((x_t,h_prev),dim=-1) @ self.W + self.b
        i,f,g,o=gates.chunk(4,dim=-1); i=torch.sigmoid(i); f=torch.sigmoid(f); g=torch.tanh(g); o=torch.sigmoid(o)
        c_t=f*c_prev+i*g; h_t=o*torch.tanh(c_t); return h_t,c_t

class MyGRUCell(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__(); self.input_dim=input_dim; self.hidden_dim=hidden_dim
        self.W=nn.Parameter(torch.empty(input_dim+hidden_dim,3*hidden_dim)); self.b=nn.Parameter(torch.zeros(3*hidden_dim)); nn.init.xavier_uniform_(self.W)
    def forward(self,x_t,h_prev):
        wxr,wxz,wxn=(x_t @ w for w in self.W[:self.input_dim].split(self.hidden_dim,dim=1)); whr,whz,whn=(h_prev @ w for w in self.W[self.input_dim:].split(self.hidden_dim,dim=1))
        r=torch.sigmoid(wxr+whr+self.b[:self.hidden_dim]); z=torch.sigmoid(wxz+whz+self.b[self.hidden_dim:2*self.hidden_dim]); n=torch.tanh(wxn+r*whn+self.b[2*self.hidden_dim:])
        return (1-z)*n+z*h_prev

def cell_parameter_counts(d=9,h=64):
    return {'RNN':h*d+h*h+h,'LSTM':4*(h*d+h*h+h),'GRU':3*(h*d+h*h+h)}
assert cell_parameter_counts()=={'RNN':4736,'LSTM':18944,'GRU':14208}
print(cell_parameter_counts())

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self,kind,input_dim=9,hidden_dim=64,classes=6):
        super().__init__(); layer={'RNN':nn.RNN,'LSTM':nn.LSTM,'GRU':nn.GRU}[kind]; self.rnn=layer(input_dim,hidden_dim,batch_first=True); self.head=nn.Linear(hidden_dim,classes)
    def forward(self,x):
        _,state=self.rnn(x); h=state[0] if isinstance(state,tuple) else state; return self.head(h[-1])

RESULT_COLUMNS=['problem','model','seed','seq_len','test_acc','macro_f1','test_mse','params','sec_per_epoch','val_loss_nats','bits_per_char','notes']
def append_result(row):
    exists=RESULTS.exists() and RESULTS.stat().st_size>0; data={k:row.get(k,'') for k in RESULT_COLUMNS}
    with RESULTS.open('a',newline='') as f:
        w=csv.DictWriter(f,fieldnames=RESULT_COLUMNS); (w.writeheader() if not exists else None); w.writerow(data)

def run_sequence(cell, x, h0=None, c0=None):
    b,t,_=x.shape; hdim=cell.hidden_dim; h=h0 if h0 is not None else x.new_zeros(b,hdim)
    c=c0 if c0 is not None else x.new_zeros(b,hdim); hs=[]
    for step in range(t):
        state=cell(x[:,step],(h,c)) if isinstance(cell,MyLSTMCell) else cell(x[:,step],h)
        h,c=state if isinstance(state,tuple) else (state,c); hs.append(h)
    return torch.stack(hs,1),(h,c)

def train_classifier(model, train_loader, val_loader, epochs=10, lr=1e-3, device=DEVICE):
    model.to(device); opt=torch.optim.Adam(model.parameters(),lr=lr); loss_fn=nn.CrossEntropyLoss(); history={'train':[],'validation':[]}; t0=time.perf_counter()
    for _ in range(epochs):
        model.train(); total=0
        for xb,yb in train_loader:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step(); total+=loss.item()*len(yb)
        model.eval(); val=0
        with torch.no_grad():
            for xb,yb in val_loader: val+=loss_fn(model(xb.to(device)),yb.to(device)).item()*len(yb)
        history['train'].append(total/len(train_loader.dataset)); history['validation'].append(val/len(val_loader.dataset))
    return history,(time.perf_counter()-t0)/max(epochs,1)

def evaluate_classifier(model, loader, device=DEVICE):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in loader: ys.extend(yb.numpy()); ps.extend(model(xb.to(device)).argmax(1).cpu().numpy())
    return {'accuracy':accuracy_score(ys,ps),'macro_f1':f1_score(ys,ps,average='macro'),'confusion_matrix':confusion_matrix(ys,ps)}

def make_loaders(train_split, validation_split=None, batch_size=128):
    validation_split=train_split if validation_split is None else validation_split
    train=TensorDataset(torch.from_numpy(train_split[0]),torch.from_numpy(train_split[1])); valid=TensorDataset(torch.from_numpy(validation_split[0]),torch.from_numpy(validation_split[1]))
    return DataLoader(train,batch_size=batch_size,shuffle=True), DataLoader(valid,batch_size=batch_size)

def plot_history(history, path):
    fig,ax=plt.subplots(); ax.plot(history['train'],label='train'); ax.plot(history['validation'],label='validation'); ax.set(xlabel='epoch',ylabel='cross-entropy'); ax.legend(); fig.savefig(path,dpi=140,bbox_inches='tight'); plt.close(fig)